# LENS-XAI: Lightweight Explainable Network Security
### End-to-End Implementation Pipeline

**Team Members:** Dan N Mecartin (23BAI1291) | Thapaswin G (23BAI1316)  
**Course:** Deep Learning  

---
This notebook contains the **entire** LENS-XAI architecture from scratch, including the Teacher/Student models, Hierarchical VAEs, Knowledge Distillation, and advanced Phase 3 & 4 capabilities (Federated Learning, EWC-based Incremental Learning, Adversarial Defense, INT8 Quantization, and cross-dataset testing).

### 1. Environment & Imports

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath('..'))

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, TensorDataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy
import time

# Ensure datasets are processed first (Run src/data/make_dataset.py if these fail)
from src.models.train import load_data
from src.utils.metrics import classification_report_extended, cross_dataset_summary, plot_confusion_matrix, plot_roc_curves

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[*] Hardware Accelerator:  {device}")

### 2. Full Model Architecture (H-VAE + Attention)
We declare the complete PyTorch architectures here. The `TeacherModel` uses Self-Attention and a Hierarchical Variational Autoencoder (H-VAE) to capture multi-scale network anomalies. The `StudentModel` is a lightweight proxy designed for Edge/IoT deployment.

In [ ]:
class SelfAttentionFeatureSelector(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(input_dim, input_dim // 2),
            nn.ReLU(),
            nn.Linear(input_dim // 2, input_dim),
            nn.Sigmoid()
        )
    def forward(self, x):
        weights = self.attention(x)
        return x * weights, weights

class HierarchicalVAE(nn.Module):
    def __init__(self, input_dim, latent_dim_1=32, latent_dim_2=16):
        super().__init__()
        # Level 1 (Packet/Flow level)
        self.enc1 = nn.Sequential(nn.Linear(input_dim, 128), nn.ReLU(), nn.BatchNorm1d(128))
        self.fc_mu1, self.fc_var1 = nn.Linear(128, latent_dim_1), nn.Linear(128, latent_dim_1)
        
        # Level 2 (Session level)
        self.enc2 = nn.Sequential(nn.Linear(latent_dim_1, 64), nn.ReLU(), nn.BatchNorm1d(64))
        self.fc_mu2, self.fc_var2 = nn.Linear(64, latent_dim_2), nn.Linear(64, latent_dim_2)
        
        self.dec2 = nn.Sequential(nn.Linear(latent_dim_2, 64), nn.ReLU(), nn.Linear(64, latent_dim_1))
        self.dec1 = nn.Sequential(nn.Linear(latent_dim_1 * 2, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Linear(128, input_dim))

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

class TeacherModel(nn.Module):
    def __init__(self, input_dim, num_classes, latent_1=32, latent_2=16):
        super().__init__()
        self.attention = SelfAttentionFeatureSelector(input_dim)
        self.h_vae = HierarchicalVAE(input_dim, latent_1, latent_2)
        self.classifier = nn.Sequential(
            nn.Linear(latent_1 + latent_2, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, num_classes)
        )
    def forward(self, x):
        weighted_x, att_w = self.attention(x)
        h1 = self.h_vae.enc1(weighted_x)
        mu1, logvar1 = self.h_vae.fc_mu1(h1), torch.clamp(self.h_vae.fc_var1(h1), -20, 5)
        z1 = self.h_vae.reparameterize(mu1, logvar1)
        h2 = self.h_vae.enc2(z1)
        mu2, logvar2 = self.h_vae.fc_mu2(h2), torch.clamp(self.h_vae.fc_var2(h2), -20, 5)
        z2 = self.h_vae.reparameterize(mu2, logvar2)
        logits = self.classifier(torch.cat([z1, z2], dim=1))
        return logits, weighted_x, att_w, mu1, logvar1, mu2, logvar2

class StudentModel(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.attention = nn.Sequential(nn.Linear(input_dim, input_dim // 4), nn.ReLU(), nn.Linear(input_dim // 4, input_dim), nn.Sigmoid())
        self.classifier = nn.Sequential(nn.Linear(input_dim, 32), nn.ReLU(), nn.Linear(32, num_classes))
    def forward(self, x):
        return self.classifier(x * self.attention(x))

### 3. All 4 Benchmark Datasets Initialization
We load all 4 evaluated network cybersecurity datasets (NSL-KDD, Edge-IIoTset, CTU-13, and UKM-IDS20).

In [ ]:
datasets = ["nsl-kdd", "edge-iiotset", "ctu-13", "ukm-ids20"]
data_loaders = {}

for ds in datasets:
    try:
        train_ldr, test_ldr, in_dim, n_cls = load_data(ds, batch_size=512)
        data_loaders[ds] = {'train': train_ldr, 'test': test_ldr, 'dim': in_dim, 'cls': n_cls}
        print(f"[+] Loaded {ds.upper():<15} | Features: {in_dim:<3} | Train Samples: {len(train_ldr.dataset):<8} | Test Samples: {len(test_ldr.dataset)}")
    except Exception as e:
        print(f"[-] Error loading {ds}: {e}")

primary_ds = "nsl-kdd"
train_loader, test_loader = data_loaders[primary_ds]['train'], data_loaders[primary_ds]['test']
input_dim, num_classes = data_loaders[primary_ds]['dim'], data_loaders[primary_ds]['cls']

### 4. Phase 1 & 2: Base KD Training
Here we run a quick 1-epoch pass of the Knowledge Distillation formulation. The Teacher trains using Ground Truth + KL Divergence across the latent spaces. The Student trains matching the Teacher's Softmax probabilities.

**Knowledge Distillation Loss:**
$$ \mathcal{L}_{KD} = \alpha \mathcal{L}_{CE}(y, \hat{y}_s) + (1-\alpha) T^2 \mathcal{D}_{KL}\left(\text{softmax}\left(\frac{z_s}{T}\right) \| \text{softmax}\left(\frac{z_t}{T}\right)\right) $$

In [ ]:
def kd_loss_function(s_logits, t_logits, labels, alpha=0.5, temp=3.0):
    ce_loss = F.cross_entropy(s_logits, labels)
    kd_loss = nn.KLDivLoss(reduction='batchmean')(F.log_softmax(s_logits/temp, dim=1), F.softmax(t_logits/temp, dim=1)) * (temp**2)
    return alpha * ce_loss + (1.0 - alpha) * kd_loss

teacher = TeacherModel(input_dim, num_classes).to(device)
student = StudentModel(input_dim, num_classes).to(device)

opt_t = optim.Adam(teacher.parameters(), lr=1e-3)
opt_s = optim.Adam(student.parameters(), lr=1e-3)

print(f"--- KD Training on {primary_ds.upper()} ---")
for data, target in train_loader:
    data, target = data.to(device), target.to(device)
    
    # 1. Train Teacher
    opt_t.zero_grad()
    t_logits, _, _, mu1, logvar1, mu2, logvar2 = teacher(data)
    cls_loss = F.cross_entropy(t_logits, target)
    kl1 = -0.5 * torch.sum(1 + logvar1 - mu1.pow(2) - logvar1.exp())
    kl2 = -0.5 * torch.sum(1 + logvar2 - mu2.pow(2) - logvar2.exp())
    (cls_loss + 0.001*(kl1+kl2)).backward()
    opt_t.step()

    # 2. Train Student
    opt_s.zero_grad()
    s_logits = student(data)
    with torch.no_grad(): t_soft = teacher(data)[0]
    s_loss = kd_loss_function(s_logits, t_soft, target)
    s_loss.backward()
    opt_s.step()
    
print("[*] 1 Epoch Distillation Complete.")

#### 4.1 Confusion Matrix Visualization
We evaluate the Student Model and plot its confusion matrix.

In [ ]:
student.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for data, target in test_loader:
        logits = student(data.to(device))
        preds = logits.max(1)[1]
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(target.numpy())
        if len(all_preds) > 5000: break

plot_confusion_matrix(np.array(all_targets), np.array(all_preds), class_names=["Normal", "Attack"], save_path="../conf_matrix.png")

from IPython.display import display, Image
display(Image(filename="../conf_matrix.png"))

### 5. Phase 3: Federated Learning
Simulating real-world decentralized node aggregation using the Federated Averaging (FedAvg) algorithm and injecting Differential Privacy (DP) Gaussian noise to the pooled weights.

**FedAvg Formula:**
$$ w_{t+1} = \sum_{k=1}^K \frac{n_k}{n} w_{t+1}^k $$

In [ ]:
from src.federated.federated_trainer import run_federated_training

print("[*] Booting 3 simulated IoT clients for Federated rounds...")
global_model = run_federated_training(n_rounds=2, n_clients=3, dataset_name=primary_ds, local_epochs=1, device=device)

### 6. Phase 3: Adversarial Robustness Layer
We construct an adversarial evasion attack to trick our Intrusion Detection System. Then, we use an `AdversarialTrainer` to perform robust optimization, allowing the system to naturally defend against manipulated feature streams.

**Fast Gradient Sign Method (FGSM):**
$$ X^{adv} = X + \epsilon \cdot \text{sign}(\nabla_X J(\theta, X, y_{true})) $$

In [ ]:
from src.models.adversarial import AdversarialTrainer

adv_trainer = AdversarialTrainer(global_model, device=device)

eval_subset = DataLoader(Subset(test_loader.dataset, range(1000)), batch_size=256)
print("--- Baseline Vulnerability (FGSM Epsilon=0.05) ---")
_ = adv_trainer.evaluate_robustness(eval_subset, epsilons=[0.0, 0.05])

print("\n--- Injecting Defense Mechanism ---")
opt = optim.Adam(global_model.parameters(), lr=1e-3)
adv_trainer.train_epoch_adversarial(eval_subset, opt, epsilon=0.05)

print("\n--- Post-Defense Robustness ---")
_ = adv_trainer.evaluate_robustness(eval_subset, epsilons=[0.0, 0.05])

### 7. Phase 3: Incremental Learning (EWC)
Real-world IDS models suffer from "catastrophic forgetting" when retrained on new zero-day attacks. By utilizing **Elastic Weight Consolidation**, the model computes a Fisher Information Matrix to freeze weights critical to past objectives while allowing redundant weights to adapt.

**EWC Loss Regularization Penalty:**
$$ \mathcal{L}_{EWC} = \mathcal{L}_{Task\ B}(\theta) + \frac{\lambda}{2} \sum_{i} F_i (\theta_i - \theta_{A,i}^*)^2 $$

In [ ]:
from src.models.incremental import IncrementalLearner

task_a_loader = DataLoader(Subset(train_loader.dataset, range(0, 500)), batch_size=256)
task_b_loader = DataLoader(Subset(train_loader.dataset, range(500, 1000)), batch_size=256)

incremental_learner = IncrementalLearner(global_model, lambda_ewc=5000, replay_buffer_size=200, device=device)

# Pre-train on Task A and freeze its Fisher Information matrix
print("--- Solidifying Knowledge into Replay Buffer (Task A) ---")
incremental_learner.consolidate(task_a_loader)

correct_initial = 0
incremental_learner.model.eval()
with torch.no_grad():
    for d, t in task_a_loader:
        d, t = d.to(device), t.to(device)
        preds = incremental_learner.model(d)[0].max(1)[1]
        correct_initial += preds.eq(t).sum().item()
        
print(f"[*] Task A Intact Baseline Accuracy: {100. * correct_initial / 500:.2f}%")

print("\n--- Training on Disrupted Set (Task B) ---")
incremental_learner.train_new_task(task_b_loader, epochs=2)

correct_retained = 0
incremental_learner.model.eval()
with torch.no_grad():
    for d, t in task_a_loader:
        d, t = d.to(device), t.to(device)
        preds = incremental_learner.model(d)[0].max(1)[1]
        correct_retained += preds.eq(t).sum().item()
        
print(f"\n[*] Task A Retained Accuracy (Post-Disruption): {100. * correct_retained / 500:.2f}%")

### 8. Phase 4: Edge Quantization & Inference Scalability
We quantize the 32-bit floating point parameters (FP32) into 8-bit integers (INT8) to allow deploying the `.pt` models to low-power IoT devices.

In [ ]:
from src.models.quantize import quantize_model, compare_model_sizes, benchmark_inference

quantized_student = quantize_model(student)
fp32_size, int8_size, red = compare_model_sizes(student, quantized_student)

print("\n--- Hardware Benchmarks (CPU Milliseconds per Batch) ---")
print("INT8 Quantized Student (Aggressive CPU Deployment):")
benchmark_inference(quantized_student, test_loader, num_batches=10, device="cpu")

### 9. Cross-Dataset Validation Summary
Evaluating the model architectures natively against all 4 distinct network cyber-attack benchmarks to prove global robustness.

In [ ]:
cross_results = {}
for ds, parts in data_loaders.items():
    print(f"[*] Evaluating zero-shot architecture on {ds.upper()}...")
    m = TeacherModel(parts['dim'], parts['cls']).to(device)
    m.eval()
    preds, tgts, probs = [], [], []
    with torch.no_grad():
        for d, t in parts['test']:
            out = m(d.to(device))[0]
            prb = torch.softmax(out, dim=1)
            _, prd = out.max(1)
            preds.extend(prd.cpu().numpy())
            tgts.extend(t.cpu().numpy())
            probs.extend(prb[:, 1].cpu().numpy() if parts['cls']==2 else prb.cpu().numpy())
            if len(preds) > 1000: break
    
    metrics, _ = classification_report_extended(tgts, preds, np.array(probs))
    cross_results[ds] = metrics

# Display the summary table
pd.DataFrame(cross_results).T

### 10. Explainable AI (XAI) Mappings
Mapping SHAP/LIME logic to actual underlying feature inputs for total model transparency.

In [ ]:
try:
    from src.utils.xai import LIMEExplainer
    bg_data = next(iter(test_loader))[0][:100]
    lime_exp = LIMEExplainer(global_model, bg_data, feature_names=[f"F_{i}" for i in range(input_dim)])
    
    # Explain the very first instance's prediction locally
    exp = lime_exp.explain_instance(bg_data[0])
    lime_exp.plot_explanation(exp, "../lime_demo.png")
    
    display(Image(filename="../lime_demo.png"))
except Exception as e:
    print("LIME execution skipped (make sure `lime` is installed).", e)